<b><font size="6" color="#E8800A">04 · Preprocessing (Nico)</font></b><br>
<b><font size="4">Missing values, outliers, encoding and scaling, each chosen by a paired comparison</font></b>

**Owner:** Nico

**Question:** how should the clean data be prepared for the models, and which choice is backed by a measurement?

**Ground rules** (the course method of Weeks 3–5)
- **Look before changing anything:** every block starts from what the data show.
- **One block of decisions at a time,** in the course order: missing values → outliers → encoding and scaling. Each block starts from the recipe the previous one fixed.
- **Every learned step is fitted on the training rows of each fold only** (`FoldPreprocessor`), so validation rows never help compute a fill, a limit or an encoding.
- **Every option is compared with the current recipe fold by fold** (`paired_delta`) and read at **two standard errors**.
- **Every decision is recorded** in `docs/decisions.md`, and the final recipe **graduates** to `src/preprocessing.py`, as rules a later notebook can reuse, not as fitted values.

**Not here:** deleting rows or columns (cleaning in `02`, feature selection in `05`), new features (`05`), model choice and tuning (`06`).

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. How a choice is verified</font>](#method)
* [<font color='#E8800A'>2. Missing values</font>](#missing)
* [<font color='#E8800A'>3. Outliers</font>](#outliers)
* [<font color='#E8800A'>4. Encoding and scaling</font>](#encoding)
* [<font color='#E8800A'>5. The final recipe</font>](#final)
* [<font color='#E8800A'>6. Graduate the recipe to `src/preprocessing.py`</font>](#graduate)
* [<font color='#E8800A'>7. Summary</font>](#summary)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

__Step 1:__ Load the clean data and the validation scheme.

`%run` executes `validation_ludovico.ipynb`, which runs `cleaning_ludovico.ipynb` first (`%%capture` hides their output). It gives back:
- `train`, `test`: the clean data;
- `X`, `y`, `dates`: features, target and purchase dates of train;
- `schemes`, `models`: the time-CV splits and the V01 baseline pipelines;
- `evaluate`, `best_threshold`: fold scoring and F1 threshold search.

In [ ]:
%%capture
%run ./validation_ludovico.ipynb

__Step 2:__ Import the course tool. `src/preprocessing.py` holds the course's `preprocessing.py` unchanged: its `FoldPreprocessor` turns a recipe (a dictionary of rules) into a transformer that is fitted inside each fold. The recipe this notebook chooses is added at the end of the same file (§6).

In [ ]:
import warnings

from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.neighbors import KNeighborsClassifier

from src.preprocessing import FoldPreprocessor

__Step 3:__ Choose the folds. By default the notebook uses the team scheme, whatever start dates `validation_ludovico` defines (`TIME_CV_STARTS`): if the team changes them, this notebook follows. The six half-year folds measured in `validation_folds_nico` are ready as an alternative: switching is one line, and every number below follows.

In [ ]:
SCHEME_STARTS = {
    "team scheme (validation_ludovico)": TIME_CV_STARTS,
    "6 half-yearly from 2022": [str(d.date()) for d in pd.date_range("2022-01-01", "2024-07-01", freq="6MS")],
}
SCHEME = "team scheme (validation_ludovico)"   # or "6 half-yearly from 2022"

splits = list(time_splits(dates, SCHEME_STARTS[SCHEME]))
NUMERIC = X.select_dtypes("number").columns.tolist()
CATEGORICAL = X.select_dtypes(["string", "object"]).columns.tolist()

print(f"{SCHEME}: {len(splits)} folds, validation rows {[len(va) for _, va in splits]}")
print(f"X: {len(X):,} rows, {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical columns")

# <font color='#E8800A'>1. How a choice is verified</font> <a class="anchor" id="method"></a>
[Back to TOC](#toc)

**The problem.** A recipe gets one score per fold, and the folds differ a lot: some periods are harder to predict whatever the recipe. The difference between two recipes is usually much smaller than the difference between two periods, so comparing two averages mixes the two effects.

**The course solution: a paired comparison** (Week 4, `paired_delta`)
1. Score both recipes on the **same folds**, so a hard period penalises both.
2. Take the difference **fold by fold**: the difficulty of each period cancels, the effect of the recipe remains.
3. The **standard error of the mean (SEM)** of those differences, their standard deviation divided by √(number of folds), says how much the mean difference would move with other folds.
4. **Verdict at two SEM:** "better" or "worse" when |mean difference| > 2 × SEM, otherwise "inside the noise". About 95% of chance fluctuations stay within two standard errors.

**The metric for comparisons: ROC AUC.** It measures how well the model ranks repeat buyers above the others, at any threshold. F1, the Kaggle metric, depends on the threshold, and `validation_folds_nico` showed that F1 at a fixed threshold misses real differences. F1 is used once, for the final estimate next to Kaggle (§5).

**The model that judges the recipe:** the logistic regression of `validation_ludovico`. A penalised linear model reacts to how its inputs are filled, transformed, encoded and scaled. A tree would hide most of these choices, because it only reads thresholds. The other families are checked at the end (§5).

__Step 4:__ `paired_delta` as in Week 4, with one change: the course compares an error (MAE, lower is better), here the score is ROC AUC (higher is better), so the sign of "better" is flipped. The ratio |dAUC| / SEM is returned too: the verdict is "better" or "worse" exactly when it is above 2, and rounded numbers can hide how close a result is to the line.

In [ ]:
def paired_delta(scores, reference):
    """Week 4 paired_delta for ROC AUC: mean fold-by-fold difference, its SEM, folds won and verdict."""
    delta = np.asarray(scores) - np.asarray(reference)
    if not delta.any():
        return {"dAUC": 0.0, "SEM": 0.0, "ratio": 0.0, "better": f"0/{len(delta)}",
                "verdict": "identical on every split"}
    sem = delta.std(ddof=1) / np.sqrt(len(delta))      # ddof=1: sample standard deviation
    verdict = ("inside the noise" if abs(delta.mean()) <= 2 * sem
               else "better" if delta.mean() > 0 else "worse")
    return {"dAUC": delta.mean(), "SEM": sem, "ratio": abs(delta.mean()) / sem,
            "better": f"{int((delta > 0).sum())}/{len(delta)}", "verdict": verdict}

__Step 5:__ The bookkeeping, as in Week 4. `report` scores a recipe, files it on a shared `board` and prints its verdict against another entry. `show` prints entries side by side. `choose` applies the decision rule declared below.

In [ ]:
board, oofs, results = {}, {}, {}   # per-fold scores, out-of-fold probabilities, summary rows


def cv_run(recipe, frame=X, model=None):
    """Score recipe + model on the folds; every step is refitted on each training fold."""
    model = LogisticRegression(max_iter=3000) if model is None else model
    pipeline = make_pipeline(FoldPreprocessor(recipe, selection=None), model)
    return evaluate(pipeline, splits, frame, y)


def report(name, recipe, frame=X, against=None, model=None):
    """Score a recipe, file it on the board and print its verdict against another board entry."""
    scores, oof = cv_run(recipe, frame, model)
    board[name], oofs[name] = scores, oof
    row = {"ROC AUC": scores["ROC AUC"].mean(), "per fold": scores["ROC AUC"].round(4).tolist()}
    line = f"{name}: ROC AUC {row['ROC AUC']:.4f} {row['per fold']}"
    if against is not None:
        row.update(paired_delta(scores["ROC AUC"], board[against]["ROC AUC"]), against=against)
        line += (f"\n    vs {against}: dAUC {row['dAUC']:+.4f} +- {row['SEM']:.4f}"
                 f" (ratio {row['ratio']:.1f}), better on {row['better']} folds -> {row['verdict'].upper()}")
    results[name] = row
    print(line)


def show(names):
    """Summary rows of these board entries, side by side."""
    return pd.DataFrame({n: results[n] for n in names}).T.style.format(
        {"ROC AUC": "{:.4f}", "dAUC": "{:+.4f}", "SEM": "{:.4f}", "ratio": "{:.1f}"}, na_rep="")


def choose(candidates, current):
    """Decision rule: the best mean among candidates judged 'better' than `current`; `current` if none."""
    better = [n for n in candidates if results[n].get("verdict") == "better" and results[n].get("against") == current]
    if not better:
        print(f"KEPT: {current} (no candidate is better beyond 2 SEM)")
        return current
    best = max(better, key=lambda n: results[n]["ROC AUC"])
    print(f"ADOPTED: {best} (better beyond 2 SEM: {better})")
    return best

<div class="alert alert-block alert-info">

**Decision rule, declared before any result.** A candidate replaces the current recipe only if its verdict is **better** (more than 2 SEM). Among several, the one with the highest mean ROC AUC wins. A tie, or a gain inside the noise, keeps the current, simpler recipe.

With 3 folds the SEM rests on 3 differences, so about one tie in six is still called a winner (`validation_folds_nico`). The ratio and the folds won are printed beside every verdict, so a borderline result can be read for what it is.

</div>

__Step 6:__ The recipe format. `FoldPreprocessor` reads a dictionary with fixed keys, the same keys the course's recipes use. Each block below changes **one** key and keeps the others.

| Key | Meaning | Starting value | Decided in |
|---|---|---|---|
| `numeric`, `categorical` | which columns enter, and as what | 28 numeric + 6 categorical | |
| `dropped` | columns kept out of the model | identifier, raw date, decoy column | `05_features` |
| `fill` | how gaps are filled | median / mode (V01 baseline) | §2 |
| `log1p`, `outliers` | outlier treatment | none, keep | §3 |
| `encoding`, `drop` | categorical encoding | one-hot, drop first | §4 |
| `scaler` | scaling | standard | §4 |
| `strategy` | feature selection | None | `05_features` |

In [ ]:
START = {
    "numeric": NUMERIC, "categorical": CATEGORICAL,
    "dropped": ["customer_id", "purchase_date", "random_noise"],
    "fill": {"numeric": "median", "categorical": "mode"},
    "log1p": [], "outliers": "keep",
    "encoding": "one-hot", "drop": "first", "scaler": "standard",
    "strategy": None,
}

__Step 7:__ Sanity check. The starting recipe applied through `FoldPreprocessor` should score like the V01 pipeline of `validation_ludovico`, which is written with a `ColumnTransformer`. Small differences are expected: V01 groups levels with fewer than 20 rows, while the course one-hot drops the first level and keeps every other one.

The **V01 pipeline** is filed on the board as the reference of the whole notebook.

In [ ]:
V01 = "V01 baseline"
scores, oof = evaluate(models["logistic regression"], splits, X, y)
board[V01], oofs[V01] = scores, oof
results[V01] = {"ROC AUC": scores["ROC AUC"].mean(), "per fold": scores["ROC AUC"].round(4).tolist()}
print(f"{V01}: ROC AUC {results[V01]['ROC AUC']:.4f} {results[V01]['per fold']}")

# Expected messages here: unseen categories and slow convergence. Both are explained in §4.2 and §4.3.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    report("start (course format)", START, against=V01)

**Reading**
- The starting recipe, applied through the course's `FoldPreprocessor`, scores **0.7800** against **0.7781** for the V01 pipeline: +0.0019 ± 0.0025, ratio 0.7, inside the noise.
- The two pipelines are equivalent, so every comparison below starts from the same place as the V01 submission (Kaggle 0.770).
- The small gap has a known source: V01 groups levels with fewer than 20 rows, while the course one-hot keeps every level but the first.

# <font color='#E8800A'>2. Missing values</font> <a class="anchor" id="missing"></a>
[Back to TOC](#toc)

> *"A missing value is unknown, not wrong."* (Week 3)

The **wrong** values are already handled: `cleaning_ludovico` turned impossible values into NaN and rebuilt the ones exact identities give back. The NaN left are **unknown**. scikit-learn models stop with an error when they meet a NaN, so every gap must be filled, and the filled value is invented. The question is which invented value does the least harm and keeps the most information.

| Kind of missing value | Meaning | What matters when filling it |
|---|---|---|
| **Scattered** | the value exists but was not recorded, by chance | invent a plausible value |
| **Structural** | the value **cannot exist** | any value is fake; what matters is that the model can tell |
| **Informative** | the **fact** that it is missing says something about the target | do not throw that fact away |

From here on, the warnings about unseen categories and convergence are silenced: §4 explains both.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)        # one-hot: unseen categories (see §4.2)
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # unscaled solver (see §4.3)

## <font color='#E8800A'>2.1 What is missing, and why</font>

__Step 8:__ Count the gaps per column, in train and in test.

In [ ]:
X_test = test[X.columns]
gaps = pd.DataFrame({"train": X.isna().sum(), "train %": (100 * X.isna().mean()).round(1),
                     "test": X_test.isna().sum()})
gaps = gaps[gaps["train"] > 0].sort_values("train", ascending=False)
print(f"rows with at least one gap: train {X.isna().any(axis=1).sum():,} of {len(X):,},"
      f" test {X_test.isna().any(axis=1).sum():,} of {len(X_test):,}")
gaps

__Step 9:__ The history columns. A gap between purchases needs at least two prior purchases, and a spread of gaps needs at least three. Count the gaps by number of prior purchases, beside the repeat rate of each group.

In [ ]:
HISTORY = ["prior_mean_gap_days", "prior_std_gap_days", "prior_std_purchase_value",
           "prior_max_purchase_value", "days_since_previous_purchase"]
depth = X["prior_purchase_count"].clip(upper=3).rename("prior purchases (3 = 3+)")
X[HISTORY].isna().groupby(depth).sum().assign(rows=depth.value_counts().sort_index(),
                                               repeat_rate=y.groupby(depth).mean().round(3))

**Reading**
- **Almost every gap in the history columns is structural.** `prior_purchase_count` says exactly which columns cannot exist:
  - 0 prior purchases (603 rows): every history column is empty;
  - 1 prior purchase (445 rows): no gap between purchases yet, so the mean gap, the spread of gaps and the spread of values are empty;
  - 2 prior purchases (365 rows): one gap only, so its spread is empty.
- **Only a handful are scattered:** 41 `prior_max_purchase_value` and 6 gap / recency values among customers with 3+ purchases.
- **The structural gaps are informative:** the repeat rate is 0.333, 0.373 and 0.400 for 0, 1 and 2 prior purchases, against 0.615 for 3+.
- **That information is already in the frame:** `prior_purchase_count` has no gaps, so the model sees it whatever the fill. §2.4 measures whether a fake filled value confuses the model anyway.

__Step 10:__ The other gaps. Does missingness move the target, and do the gaps fall on the same rows?

In [ ]:
SCATTERED = ["payment_type", "salesperson", "transport_method", "warehouse", "spend_365d"]
missingness = pd.DataFrame({col: {"repeat rate if missing": y[X[col].isna()].mean(),
                                  "repeat rate if present": y[X[col].notna()].mean(),
                                  "rows missing": X[col].isna().sum()} for col in SCATTERED}).T
per_row = X[SCATTERED[:4]].isna().sum(axis=1)
print(f"rows with a gap in one of the 4 categoricals: {(per_row >= 1).sum()}, in two or more: {(per_row >= 2).sum()}")
missingness.round(3)

**Reading**
- **These gaps are scattered:** 749 rows have a gap in one of the four categoricals, and only 31 in two or more. The gaps fall on different rows, as if injected one column at a time.
- **For `payment_type` and `transport_method` missingness says nothing** (0.564 and 0.551 against 0.560).
- `salesperson`, `warehouse` and `spend_365d` show a higher repeat rate when missing (0.598, 0.636, 0.647 against about 0.56), on fewer than 200 rows each, which may be chance. An *own level* fill lets the model use it if it is real, without assuming it.

## <font color='#E8800A'>2.2 The fill plans</font>

**Two rules from Week 3**
1. **A fill is fitted on the training rows and applied to both halves.** A median taken over the whole file would be partly taken from the rows the model is scored on: that is leakage. `FoldPreprocessor` refits the fill inside every fold.
2. **KNN and iterative imputation work on standardised columns**, then return their fills in the original units. KNN finds neighbours by distance, so without standardising, a column in tens of thousands (`prior_total_spend`) would choose the neighbours alone.

| Fill for numeric gaps | Becomes | Assumes |
|---|---|---|
| `median` | the training median | the typical value is a fair guess; unmoved by extreme values |
| `mean` | the training mean | the same, but the mean is pulled by long tails |
| `zero` | 0 | "missing" means "none": true for some columns, absurd for others |
| `KNN` (k=5) | mean of the 5 most similar training rows | similar customers on the other columns are similar on this one |
| `iterative` | each column predicted from the others, repeated | the same, through linear relations |

| Fill for categorical gaps | Becomes | Assumes |
|---|---|---|
| `own level` | a new level, `(missing)` | nothing is guessed: the model learns what "missing" is worth |
| `mode` | the most frequent training level | the most common case is the best guess |
| `proportional` | a level drawn at random in its training shares | keeps the mix of levels, but adds random noise |

**The plans** are the nine of Week 3, named *numeric / categorical*, with **median / own level** as the reference, as in the course. Two coincide with the reference on our data and are not scored:
- **skew-aware / own level** (median on skewed columns, mean on the others): the cell below shows how many columns are skewed.
- **median / missing left to encode:** the encoder then treats a NaN as a level of its own, which is exactly *own level*.

**median / training mode** is the plan of the V01 submission.

In [ ]:
skewed = X[NUMERIC].skew() > 1
print(f"numeric columns with skew > 1: {skewed.sum()} of {len(NUMERIC)}; the others: {skewed[~skewed].index.tolist()}")

FILL_PLANS = {
    "median / own level": {"numeric": "median", "categorical": "own level"},   # course reference
    "mean / own level": {"numeric": "mean", "categorical": "own level"},
    "zero / own level": {"numeric": "zero", "categorical": "own level"},
    "median / training mode": {"numeric": "median", "categorical": "mode"},    # V01 submission
    "median / proportional": {"numeric": "median", "categorical": "proportional", "random_state": RANDOM_STATE},
    "KNN (k=5) / own level": {"numeric": "KNN", "n_neighbors": 5, "standardised": True, "categorical": "own level"},
    "iterative / own level": {"numeric": "iterative", "standardised": True, "random_state": RANDOM_STATE,
                              "categorical": "own level"},
}
FILL_REFERENCE = "median / own level"

## <font color='#E8800A'>2.3 The board</font>

__Step 11:__ Score the seven plans on the same folds, each against the reference. Only the `fill` key changes.

In [ ]:
fill_names = [f"fill · {name}" for name in FILL_PLANS]
reference = f"fill · {FILL_REFERENCE}"
report(reference, {**START, "fill": FILL_PLANS[FILL_REFERENCE]})
for name in FILL_PLANS:
    if name != FILL_REFERENCE:
        report(f"fill · {name}", {**START, "fill": FILL_PLANS[name]}, against=reference)

In [ ]:
show(fill_names)

__Step 12:__ The board as a picture: each candidate's difference from the reference, fold by fold (dots), and its mean with ± 2 SEM (bar). A bar that does not touch 0 is a real difference.

In [ ]:
def plot_deltas(names, reference, title):
    """Fold-by-fold differences from `reference` (dots) and mean +- 2 SEM (bars)."""
    names = sorted(names, key=lambda n: results[n]["dAUC"])
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(names) + 1.5))
    for i, name in enumerate(names):
        delta = board[name]["ROC AUC"].to_numpy() - board[reference]["ROC AUC"].to_numpy()
        ax.scatter(delta, [i] * len(delta), color=PLOT_BLUE, s=16, alpha=0.6,
                   label="single folds" if i == 0 else None)
        ax.errorbar(results[name]["dAUC"], i, xerr=2 * results[name]["SEM"], fmt="o",
                    color=PLOT_ORANGE, capsize=4, label="mean ± 2 SEM" if i == 0 else None)
    ax.axvline(0, color="grey", lw=1)
    ax.set_yticks(range(len(names)), [n.split(" · ", 1)[1] for n in names])
    ax.set(xlabel=f"ROC AUC difference from {reference.split(' · ', 1)[1]}", title=title)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


plot_deltas([n for n in fill_names if n != reference], reference, "Fill plans against the reference")

In [ ]:
fill_choice = choose(fill_names, reference)

**Reading**
- **Two plans are better than the reference beyond two SEM:**
  - **zero / own level:** +0.0008 ± 0.0002, ratio 5.1;
  - **iterative / own level:** +0.0010 ± 0.0004, ratio 2.5.
  The rule adopts **iterative / own level**, the higher mean of the two.
- **KNN has the highest mean (+0.0014) but stays inside the noise:** ratio 1.8. Its gain is uneven across folds, as the plot shows, so its SEM is larger. This is why the ratio is printed: the rounded +0.0014 ± 0.0008 does not show how close to the line it is.
- **The V01 plan (median / training mode)** is −0.0011 ± 0.0014, inside the noise: filling a missing label with the most common level did not hurt V01 measurably.
- **The real lesson is the size.** Every plan is within about ±0.0015 ROC AUC of the reference. In Week 3 the KNN fill gained about 1.2% of the error on the professor's data; here the best fill gains about 0.1%. Our gaps carry little that the other columns do not already say, so **the fill is not a lever for the score.**
- Why could *zero* help? For the structural gaps, 0 is outside the real range of a gap (at least 1 day), so it marks "no history" with a value no real customer has. This is a hypothesis, not something the board proves.

## <font color='#E8800A'>2.4 Structural gaps as information</font>

A fill gives a first purchase the gap of a typical customer: a fake value that looks like a regular buyer. The model also sees `prior_purchase_count`, so it may already know the value is fake. Whether it does is measured, not assumed.

`FoldPreprocessor` has no missing-indicator option, so the information is added the way Week 4 adds `km_per_year`: as **constructed columns**, defined by a formula and judged by a paired delta. The formula reads only the row itself, so it fits nothing and cannot leak.

__Step 13:__ Three 0/1 flags for the history depth that decides which columns are structurally empty, added to the plan chosen in §2.3.

In [ ]:
FLAGS = ["no_prior_purchase", "one_prior_purchase", "two_prior_purchases"]


def add_history_flags(frame):
    """0/1 flags for 0, 1 and 2 prior purchases: the depths that leave history columns empty."""
    count = frame["prior_purchase_count"]
    return frame.assign(no_prior_purchase=(count == 0).astype(int),
                        one_prior_purchase=(count == 1).astype(int),
                        two_prior_purchases=(count == 2).astype(int))


fill_plan = FILL_PLANS[fill_choice.split(" · ", 1)[1]]
flags_name = "fill · chosen plan + history flags"
report(flags_name, {**START, "fill": fill_plan, "numeric": NUMERIC + FLAGS},
       frame=add_history_flags(X), against=fill_choice)
flags_added = results[flags_name]["verdict"] == "better"

**Reading:** the three history-depth flags change the score by −0.0002 ± 0.0008 (ratio 0.2) and win 1 fold of 3: inside the noise, so they are **not added**. The model already knows the depth from `prior_purchase_count`, so a separate flag adds nothing for a linear model. `05_features` can test them again with tree models.

## <font color='#E8800A'>2.5 Decision</font>

__Step 14:__ Fix the fill in the recipe.

In [ ]:
RECIPE = {**START, "fill": fill_plan}
FRAME = X
if flags_added:
    RECIPE["numeric"] = NUMERIC + FLAGS
    FRAME = add_history_flags(X)

current = "after §2 · missing values"
report(current, RECIPE, FRAME, against=V01)
print(f"\nfill: {RECIPE['fill']} | history flags added: {flags_added}")

# <font color='#E8800A'>3. Outliers</font> <a class="anchor" id="outliers"></a>
[Back to TOC](#toc)

An outlier is a value far from the others in its column. Far is not the same as wrong (Week 3, *"An outlier is not always a defect"*):

| Situation | Example | What is done |
|---|---|---|
| **Error** | −4 days, a spend of 10⁹ | fixed or set to NaN: the job of **cleaning**, already done |
| **Extreme but real** | a wholesaler who spent 250,000 € | **kept**: a real customer, often the most important |
| **Long tail** | most invoices under 1,000 €, a few far above | the **shape** of the column is changed, no row is removed |

**The course tools**
- **The 1.5 × IQR rule screens, it never decides.** IQR = Q3 − Q1; a value below Q1 − 1.5·IQR or above Q3 + 1.5·IQR is flagged. It raises the question "who are these rows?", answered by looking at them.
- **`log1p`** computes log(1 + x): it compresses a long right tail, keeps the order of the values and deletes nothing. It fits nothing, so it cannot leak. Recipe key: `log1p`.
- **Winsorize 1/99** caps every value at the 1st and 99th percentiles of the training rows of the fold. Recipe key: `outliers: "winsorize 1/99"`. Deleting rows is not an option of `FoldPreprocessor`: a transformer cannot remove training rows.
- **A transform is chosen per column, by a paired delta** (Week 4: `log1p` on `km_per_year` gave back its whole gain). So `log1p` is scored group by group, not only on everything at once.

## <font color='#E8800A'>3.1 Screening</font>

__Step 15:__ Apply the IQR rule to every numeric column, and compare the flagged rows with the others.

In [ ]:
q1, q3 = X[NUMERIC].quantile(0.25), X[NUMERIC].quantile(0.75)
iqr = q3 - q1
outside = (X[NUMERIC] < q1 - 1.5 * iqr) | (X[NUMERIC] > q3 + 1.5 * iqr)
flagged = outside.any(axis=1)

print(f"rows flagged on at least one column: {flagged.sum():,} of {len(X):,} ({flagged.mean():.0%})")
print(f"repeat rate: flagged rows {y[flagged].mean():.3f}, other rows {y[~flagged].mean():.3f}")
print(f"columns with IQR = 0 (every non-typical value is 'outside'): {iqr[iqr == 0].index.tolist()}")

pd.DataFrame({"rows flagged": outside.sum(), "IQR": iqr, "% zeros": 100 * (X[NUMERIC] == 0).mean(),
              "skew": X[NUMERIC].skew(), "skew after log1p": np.log1p(X[NUMERIC].clip(lower=0)).skew()}
             ).sort_values("rows flagged", ascending=False).round(2)

__Step 16:__ Look at who the most extreme rows are: the top 1% of three columns that describe how big and how loyal a customer is.

In [ ]:
extremes = {}
for col in ["prior_total_spend", "total_net", "purchase_count_365d"]:
    top = X[col] >= X[col].quantile(0.99)
    extremes[col] = {"rows": int(top.sum()), "distinct customers": train.loc[top, "customer_id"].nunique(),
                     "repeat rate": round(y[top].mean(), 3), "column max": X[col].max()}
pd.DataFrame(extremes).T

**Reading**
- **The IQR rule would flag 66% of the training rows.** Three columns have an IQR of zero (`total_shipping`, `discount_global`, `invoice_count_current_purchase`): they are zero (or one) in at least three rows out of four, so every other value counts as "outside". It is the same failure as the `tax` column of Week 3.
- **The flagged rows are the active customers, not errors:** their repeat rate is 0.603 against 0.476.
- **The top 1% are a few big, loyal customers:**
  - the top 1% of lifetime spend is 4 customers, repeat rate 0.862;
  - the top 1% of purchases in the last year is a single customer, repeat rate 0.988.
- **No row is deleted.** The cleaning already removed impossible values. What remains are long tails of real customers, which is a question about the shape of the columns, treated below.

## <font color='#E8800A'>3.2 The board</font>

__Step 17:__ Score the treatments against the recipe of §2. The groups follow the column families of the EDA; inside each group, `log1p` goes only on the columns with skew > 1. The skew is read from the features alone, never from the target.

In [ ]:
SKEWED = [c for c in NUMERIC if X[c].skew() > 1]
GROUPS = {
    "current amounts": ["total_to_pay", "total_gross", "total_net", "total_tax", "total_shipping",
                        "discount_global", "discount_line", "cost"],
    "history amounts": ["prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
                        "prior_min_purchase_value", "prior_max_purchase_value"],
    "rolling spend": [f"spend_{w}d" for w in (30, 90, 180, 365)],
    "counts": ["n_lines", "invoice_count_current_purchase", "prior_purchase_count"]
              + [f"purchase_count_{w}d" for w in (30, 90, 180, 365)],
    "day intervals": ["days_since_previous_purchase", "prior_mean_gap_days", "prior_std_gap_days"],
}
assert sorted(sum(GROUPS.values(), [])) == sorted(c for c in NUMERIC if c != "customer_tenure_days"), \
    "every numeric column but customer_tenure_days (not skewed) belongs to one group"

TREATMENTS = {
    "winsorize 1/99": {"outliers": "winsorize 1/99"},
    "log1p · all skewed": {"log1p": SKEWED},
    "winsorize + log1p · all skewed": {"outliers": "winsorize 1/99", "log1p": SKEWED},
    **{f"log1p · {group}": {"log1p": [c for c in cols if c in SKEWED]} for group, cols in GROUPS.items()},
}

outlier_names = []
for treatment, change in TREATMENTS.items():
    name = f"outliers · {treatment}"
    report(name, {**RECIPE, **change}, FRAME, against=current)
    outlier_names.append(name)

__Step 18:__ If more than one group passes on its own, their columns are combined and scored once more, as Week 4 combines what each family keeps.

In [ ]:
passing = [n for n in outlier_names if "log1p · " in n and "all skewed" not in n
           and results[n]["verdict"] == "better"]
if len(passing) > 1:
    combined = sorted({c for n in passing for c in TREATMENTS[n.split(" · ", 1)[1]]["log1p"]})
    TREATMENTS["log1p · passing groups"] = {"log1p": combined}
    report("outliers · log1p · passing groups", {**RECIPE, "log1p": combined}, FRAME, against=current)
    outlier_names.append("outliers · log1p · passing groups")
print(f"groups better on their own: {passing}")
show(outlier_names)

In [ ]:
plot_deltas(outlier_names, current, "Outlier treatments against the recipe of §2")

In [ ]:
outlier_choice = choose(outlier_names, current)

**Reading**
- **Winsorize is clearly worse:** −0.0031 ± 0.0006, worse on every fold. Capping the top values removes information exactly where the biggest and most loyal customers are.
- **`log1p` on every skewed column at once is about zero** (−0.0001), because gains and losses of the single groups cancel out. Group by group the picture is different, which is Week 4's lesson that a transform is chosen per column:
  - **day intervals:** +0.0008 ± 0.0003, better on every fold, ratio 3.1: **adopted**;
  - **counts:** −0.0048 ± 0.0016, worse on every fold;
  - **history amounts:** −0.0018 ± 0.0010, worse on every fold, ratio 1.9: just inside the noise.
- **A possible reason (interpretation, not proof).** A logistic regression assumes that each unit of a column moves the odds by the same amount.
  - Going from 10 to 60 days since the last purchase matters much more than going from 500 to 550, and the log scale says exactly that, so it helps.
  - A customer with 60 purchases a year repeats 99% of the time. The log pulls that customer close to one with 10 purchases, and the model loses part of the distinction, so it hurts.
- `log1p` on the current amounts (+0.0013) wins only 1 fold of 3 and is not adopted.

## <font color='#E8800A'>3.3 Decision</font>

__Step 19:__ Fix the outlier treatment in the recipe.

In [ ]:
if outlier_choice != current:
    RECIPE = {**RECIPE, **TREATMENTS[outlier_choice.split(" · ", 1)[1]]}

current = "after §3 · outliers"
report(current, RECIPE, FRAME, against=V01)
print(f"\nlog1p: {RECIPE['log1p']} | outliers: {RECIPE['outliers']}")

# <font color='#E8800A'>4. Encoding and scaling</font> <a class="anchor" id="encoding"></a>
[Back to TOC](#toc)

A model reads numbers only, so every categorical level must become a number. Week 4 chooses the **encoder and the scaler together**, crossing every encoder with every scaler. Ordinal, count and target encoding turn a level into a number, and that number is scaled like every other; one-hot indicators stay 0/1. The two choices interact, so they are measured together.

| Encoder | A level becomes | Assumes / risks |
|---|---|---|
| **one-hot** (`drop="first"`) | one 0/1 column per level, the first level dropped as reference | no order between levels; many rare columns when a variable has many levels |
| **ordinal** | an integer per level | an **artificial order**: level 8 is "more" than level 3 |
| **count** | the share of training rows with that level | a level is worth its frequency; two levels with the same frequency become identical |
| **target** | the smoothed repeat rate of that level in training | reads the target, so it is fitted **inside the fold only** |

| Scaler | Does |
|---|---|
| none | raw units |
| standard | mean 0, standard deviation 1 |
| min-max | range [0, 1] |
| robust | median 0, IQR 1: less sensitive to extreme values |

## <font color='#E8800A'>4.1 The categorical columns</font>

__Step 20:__ For each categorical column: number of levels, share of the dominant level, rare levels, and levels that appear in test but never in train.

In [ ]:
rows = []
for col in CATEGORICAL:
    counts = X[col].value_counts()
    unseen = X_test[col].notna() & ~X_test[col].isin(counts.index)
    rows.append({"column": col, "levels": counts.size, "top level share": round(counts.iloc[0] / counts.sum(), 3),
                 "levels with < 20 rows": int((counts < 20).sum()),
                 "test rows with an unseen level": int(unseen.sum()),
                 "unseen levels": sorted(X_test.loc[unseen, col].unique())})
pd.DataFrame(rows).set_index("column")

__Step 21:__ The same question inside the validation scheme: how many validation rows of each fold carry a level that the fold's training rows never saw? The answer says whether the folds can reproduce the situation of the test.

In [ ]:
unseen_by_fold = pd.DataFrame({
    f"fold {k}": {col: int((X[col].iloc[va].notna() & ~X[col].iloc[va].isin(set(X[col].iloc[tr].dropna()))).sum())
                  for col in CATEGORICAL}
    for k, (tr, va) in enumerate(splits, 1)
})
unseen_by_fold["test (vs all train)"] = [int((X_test[c].notna() & ~X_test[c].isin(set(X[c].dropna()))).sum())
                                         for c in CATEGORICAL]
unseen_by_fold

**Reading**
- **`transport_method` has 80 levels, 46 of them with fewer than 20 rows.** One-hot turns it into about 75 columns that are almost always zero.
- **The test contains levels never seen in train:** 209 rows of `commercial_zone` (21% of the test: Zone 1 and Zone 5), 63 of `transport_method`, 7 of `warehouse`, 3 of `payment_type`.
- **No fold contains an unseen `commercial_zone` level:** the folds see unseen levels of `transport_method`, `payment_type` and `salesperson`, but never of `commercial_zone`. The cross-validation therefore cannot measure what happens to those 209 test rows; §4.2 looks at it directly.

## <font color='#E8800A'>4.2 What each encoder does with an unseen level</font>

__Step 22:__ Fit each encoder on all train, transform the test rows, and read how every `commercial_zone` level comes out. With `drop="first"`, the reference level is encoded as "all zeros", and `handle_unknown="ignore"` also encodes an unseen level as "all zeros". This is the warning scikit-learn prints for this case, silenced above.

In [ ]:
def encoding_of(encoder, column):
    """How each level of `column` in test comes out of `encoder` fitted on all train."""
    recipe = {**RECIPE, "categorical": [column], "encoding": encoder, "scaler": "none", "log1p": [],
              "fill": {"numeric": "median", "categorical": "own level"}}
    fitted = FoldPreprocessor(recipe, selection=None).fit(X, y)
    names = list(fitted.get_feature_names_out())
    width = len(names) - len(recipe["numeric"])            # the categorical block comes first
    encoded = pd.DataFrame(fitted.transform(X_test)[:, :width], columns=names[:width], index=X_test.index)
    encoded.columns = [c.replace(f"{column}_", "") for c in encoded.columns]
    out = encoded.groupby(X_test[column].values).first()
    out.insert(0, "seen in train", out.index.isin(X[column].dropna().unique()))
    out.insert(1, "test rows", X_test[column].value_counts().reindex(out.index).values)
    return out.round(3)


for encoder in ["one-hot", "count", "target"]:
    print(f"--- {encoder}")
    display(encoding_of(encoder, "commercial_zone"))
print("repeat rate per commercial_zone in train:", y.groupby(X["commercial_zone"]).mean().round(3).to_dict())

**Reading**
- **One-hot encodes the 189 Zone 1 rows and the 20 Zone 5 rows exactly like Zone 2**, the dropped reference level: all zeros. In training Zone 2 has a repeat rate of 0.411, against 0.565 for Zone 4, the most common zone. 209 test rows would be treated as members of a low-repeat zone they do not belong to.
- **Count encoding turns an unseen level into 0:** "never seen in training", a value no seen level has.
- **Target encoding turns it into 0.560,** the overall repeat rate: "nothing is known".
- This is the warning scikit-learn prints when `drop="first"` meets `handle_unknown="ignore"`, silenced at the start of §2.

## <font color='#E8800A'>4.3 The board: 16 combinations</font>

__Step 23:__ Every encoder crossed with every scaler, on the recipe of §3, each against the current recipe (one-hot + standard). Unscaled recipes may print convergence messages: the solver needs more steps when columns have very different scales, which is itself part of what is measured.

In [ ]:
ENCODERS = ["one-hot", "ordinal", "count", "target"]
SCALERS = ["none", "standard", "min-max", "robust"]
encoding_names = []
for encoder in ENCODERS:
    for scaler in SCALERS:
        name = f"encoding · {encoder} + {scaler}"
        if (encoder, scaler) == (RECIPE["encoding"], RECIPE["scaler"]):
            board[name], oofs[name], results[name] = board[current], oofs[current], {
                **results[current], **paired_delta(board[current]["ROC AUC"], board[current]["ROC AUC"]),
                "against": current}
            print(f"{name}: the current recipe")
        else:
            report(name, {**RECIPE, "encoding": encoder, "scaler": scaler}, FRAME, against=current)
        encoding_names.append(name)

In [ ]:
grid = pd.DataFrame({e: {s: results[f"encoding · {e} + {s}"]["ROC AUC"] for s in SCALERS} for e in ENCODERS})
grid.style.format("{:.4f}").background_gradient(cmap="Blues", axis=None)

In [ ]:
show(encoding_names)

In [ ]:
encoding_by_rule = choose(encoding_names, current)

**Reading**
- **No combination is better beyond 2 SEM,** so the rule keeps one-hot + standard.
- **Count encoding with standard or robust scaling has the highest means** (0.7887 and 0.7889, +0.006) and **wins on every fold.** The ratio is only 1.2–1.3 because the gain is uneven: small on folds 1 and 2, large on fold 3 (0.789 against 0.774).
- **Min-max is the worst scaler for one-hot** (−0.0043, worse on every fold): it squeezes the numeric columns into [0, 1] beside 0/1 dummies, and the penalty of the logistic regression is spread badly among columns.
- **Ordinal is unstable across folds** (from 0.772 to 0.789): the order it invents between levels helps in one period and hurts in another.
- **Standard and robust scaling are equivalent** in every row, so standard is kept as the scaler.
- **The unscaled rows ("none") differ slightly from run to run:** without scaling the solver does not fully converge, and where it stops depends on the machine. Another reason to scale.

__Step 24:__ Diagnostic only, not a candidate: remove one categorical column at a time from the current recipe, to see which one drives the fold-by-fold pattern. Removing columns is a feature-selection decision and belongs to `05_features`.

In [ ]:
for col in CATEGORICAL:
    report(f"diagnostic · without {col}", {**RECIPE, "categorical": [c for c in CATEGORICAL if c != col]},
           FRAME, against=current)

**Reading**
- **`transport_method` drives the fold-3 pattern.** Without it the recipe scores 0.786 on fold 3 against 0.774 with it (+0.012), while the other folds change little. Its ~75 rare one-hot columns let the model fit levels with very few rows, and those weights do not hold in the most recent period.
- `salesperson` and `warehouse` carry real signal: removing either is worse on every fold.
- Removing columns is a feature-selection decision, so nothing is dropped here. The finding goes to `05_features`.

## <font color='#E8800A'>4.4 Decision</font>

The cross-validation cannot measure the problem shown in §4.2: no fold contains a `commercial_zone` level unseen in its training rows, while the test does. Week 5: a validation estimate is trustworthy only where the validation and the test ask the same question. So the encoder is decided in two steps:
1. **Read the board:** if no encoder is better beyond 2 SEM, the board does not separate them.
2. **Among encoders the board cannot separate,** choose the one that treats a level never seen in training correctly. Count encoding turns it into 0, "never seen". One-hot with `drop="first"` turns it into the reference level, a real level with its own repeat rate.

__Step 25:__ Apply the two steps.

In [ ]:
CANDIDATE = "encoding · count + standard"
check = results[CANDIDATE]
print(f"rule on the board: {encoding_by_rule}")
print(f"{CANDIDATE}: dAUC {check['dAUC']:+.4f} +- {check['SEM']:.4f} (ratio {check['ratio']:.1f}),"
      f" better on {check['better']} folds -> {check['verdict']}")

if encoding_by_rule != current:
    encoding_choice = encoding_by_rule                       # the board decides
elif check["verdict"] != "worse":
    encoding_choice = CANDIDATE                              # board cannot separate: unseen levels decide
else:
    encoding_choice = current                                # count is measurably worse: keep one-hot
print(f"CHOSEN: {encoding_choice}")

In [ ]:
encoder, scaler = encoding_choice.split(" · ", 1)[1].split(" + ")
RECIPE = {**RECIPE, "encoding": encoder, "scaler": scaler, "drop": "first" if encoder == "one-hot" else None}

current = "after §4 · encoding and scaling"
report(current, RECIPE, FRAME, against=V01)
print(f"\nencoding: {RECIPE['encoding']} | scaler: {RECIPE['scaler']}")

**Reading**
- The board alone keeps one-hot + standard: no encoder is better beyond 2 SEM.
- **Count + standard is not worse** (+0.0059 ± 0.0048, ratio 1.2, better on all 3 folds), so the second branch of the rule decides: **count + standard**, because it treats a level never seen in training as "never seen" instead of as the reference level.
- Count encoding also turns each categorical into **one** column (6 in total instead of about 100), which removes the rare `transport_method` dummies found in §4.3.
- The cost: two levels with the same frequency become indistinguishable.

# <font color='#E8800A'>5. The final recipe</font> <a class="anchor" id="final"></a>
[Back to TOC](#toc)

__Step 26:__ The recipe the three blocks chose, against the starting recipe and against the V01 pipeline.

In [ ]:
FINAL, FINAL_FRAME = RECIPE, FRAME
for key in ["fill", "log1p", "outliers", "encoding", "drop", "scaler", "strategy"]:
    print(f"  {key:9s} {FINAL[key]}")
print(f"  columns   {len(FINAL['numeric'])} numeric, {len(FINAL['categorical'])} categorical\n")

for against in ["start (course format)", V01]:
    r = paired_delta(board[current]["ROC AUC"], board[against]["ROC AUC"])
    print(f"final vs {against}: dAUC {r['dAUC']:+.4f} +- {r['SEM']:.4f} (ratio {r['ratio']:.1f}),"
          f" better on {r['better']} folds -> {r['verdict'].upper()}")

__Step 27:__ The honest F1, as in `validation_ludovico` and the V01 submission: the threshold is chosen on the earlier folds and the F1 is scored on the last fold, the one closest to the test. This is the number that goes next to a Kaggle score.

In [ ]:
def honest_f1(name):
    """F1 on the last fold at the threshold that maximises F1 on the earlier folds."""
    oof = oofs[name]
    early = np.concatenate([va for _, va in splits[:-1]])
    last = splits[-1][1]
    threshold, _ = best_threshold(y.iloc[early], oof.iloc[early])
    return f1_score(y.iloc[last], oof.iloc[last] >= threshold), threshold


last = splits[-1][1]
floor = f1_score(y.iloc[last], np.ones(len(last)))
pd.DataFrame({name: dict(zip(["honest F1", "threshold (earlier folds)"], honest_f1(name)),
                         **{"F1 always 1 (last fold)": floor})
              for name in [V01, "start (course format)", current]}).T.style.format("{:.3f}")

__Step 28:__ The recipe was chosen through a logistic regression. Does it hold for other model families? An untuned gradient boosting (trees: insensitive to scaling and `log1p`) and an untuned KNN (distances: very sensitive to scaling), each with the starting and the final recipe.

In [ ]:
OTHER_MODELS = {"gradient boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
                "KNN (k=25)": KNeighborsClassifier(n_neighbors=25)}
for model_name, model in OTHER_MODELS.items():
    report(f"{model_name} · start", START, X, model=clone(model))
    report(f"{model_name} · final", FINAL, FINAL_FRAME, against=f"{model_name} · start", model=clone(model))

**Reading**
- **Against the V01 pipeline the final recipe is better on every fold:** +0.0106 ± 0.0029, ratio 3.7. Against the starting recipe: +0.0087 ± 0.0050, on every fold but with a ratio of 1.7.
- **The honest F1 rises from 0.771 to 0.787,** against a floor of 0.754 for "always 1": from +0.017 to +0.033 above the floor. It is the first time the Kaggle metric moves: the threshold drops from 0.37 to 0.33, so the model predicts 1 more often and gains in recall.
- **Gradient boosting improves too:** +0.0067 ± 0.0018, better on every fold. Trees ignore scaling and `log1p`, so this gain comes from the encoding.
- **KNN gets worse:** −0.0231 ± 0.0139 (inside the noise, but large). A likely reason, to be verified in `06_models`: standard scaling divides each count-encoded column by its spread. For a column with one dominant level, such as `warehouse` (99.7% one level), the spread is tiny, so a row with the rare level gets a very large value and dominates the distances. With one-hot, the same information was a 0/1 dummy.
- **Consequence for `06_models`:** the recipe was chosen through a logistic regression. It holds for trees, but distance-based models may need their own scaler or encoder.

# <font color='#E8800A'>6. Graduate the recipe to `src/preprocessing.py`</font> <a class="anchor" id="graduate"></a>
[Back to TOC](#toc)

The repository keeps experiments in notebooks, the code later notebooks reuse in `src/`, and the decisions in `docs/decisions.md`. The recipe chosen here therefore **graduates** to `src/preprocessing.py` as `final_recipe()`, beside the course's `FoldPreprocessor`. `05_features`, `06_models` and the final notebook import it instead of retyping it.

The recipe is a set of **rules, not fitted values**: medians, imputation models, encoder frequencies and scaler limits are refitted on whatever training rows the next notebook uses (Week 4: *carry the strategy, not the columns*).

__Step 29:__ Check that the recipe in `src/preprocessing.py` is exactly the one chosen in this notebook. If a block above chooses differently (other folds, other data), this cell fails, and `final_recipe()` must be updated before anything is committed.

In [ ]:
from src.preprocessing import final_recipe

assert final_recipe(X) == FINAL, "src/preprocessing.py does not hold the recipe chosen in this notebook"
print("src/preprocessing.py · final_recipe() matches the recipe chosen in this notebook")

__Step 30:__ The rows for `docs/decisions.md`, built from the board so that every number is the one measured above.

In [ ]:
def evidence(name, against):
    """Paired dAUC, SEM, folds won and verdict of `name` against `against`."""
    r = paired_delta(board[name]["ROC AUC"], board[against]["ROC AUC"])
    return f"{r['dAUC']:+.4f} +- {r['SEM']:.4f} ROC AUC, {r['better']} folds ({r['verdict']})"


decision_rows = pd.DataFrame([
    {"Decision": f"Fill numeric gaps with {FINAL['fill']['numeric']}, categorical gaps with {FINAL['fill']['categorical']}, inside each fold",
     "Alternatives considered": "median, mean, zero, KNN (k=5); mode, proportional (Week 3 plans)",
     "Evidence": f"preprocessing_nico §2.3: {evidence(fill_choice, reference)} vs {FILL_REFERENCE}"},
    {"Decision": "History-depth flags " + ("added" if flags_added else "not added"),
     "Alternatives considered": "3 flags for 0 / 1 / 2 prior purchases",
     "Evidence": f"§2.4: {evidence(flags_name, fill_choice)}"},
    {"Decision": "No row deleted for outliers; the 1.5×IQR rule only screens",
     "Alternatives considered": "delete IQR outliers",
     "Evidence": f"§3.1: flags {flagged.mean():.0%} of train rows, repeat rate {y[flagged].mean():.2f} vs {y[~flagged].mean():.2f}"},
    {"Decision": f"log1p on {', '.join(FINAL['log1p']) or 'no column'}",
     "Alternatives considered": "none, all skewed, by group, winsorize 1/99",
     "Evidence": f"§3.2: {evidence(outlier_choice, 'after §2 · missing values')}" if outlier_choice != "after §2 · missing values" else "§3.2: no treatment better"},
    {"Decision": f"{FINAL['encoding']} encoding + {FINAL['scaler']} scaling",
     "Alternatives considered": "16 encoder × scaler combinations (Week 4)",
     "Evidence": f"§4: {evidence(encoding_choice, 'after §3 · outliers')}; one-hot encodes unseen test levels as the reference level"
                 if encoding_choice != "after §3 · outliers" else "§4: no combination better"},
    {"Decision": "Final recipe (src/preprocessing.py · final_recipe)",
     "Alternatives considered": "V01 pipeline",
     "Evidence": f"§5: {evidence(current, V01)}; honest F1 {honest_f1(current)[0]:.3f} vs {honest_f1(V01)[0]:.3f}"},
])
decision_rows.insert(0, "Date", pd.Timestamp.today().strftime("%Y-%m-%d"))
decision_rows["Who"] = "Nico"

# Ready to paste under the table header of docs/decisions.md
for row in decision_rows.itertuples(index=False):
    print("| " + " | ".join(row) + " |")

# <font color='#E8800A'>7. Summary</font> <a class="anchor" id="summary"></a>
[Back to TOC](#toc)

__Step 31:__ Every decision of the notebook, with its evidence, read from the board.

In [ ]:
def line(name, against):
    r = paired_delta(board[name]["ROC AUC"], board[against]["ROC AUC"])
    return f"{r['dAUC']:+.4f} +- {r['SEM']:.4f}, {r['better']} folds ({r['verdict']})"


pd.DataFrame([
    {"block": "missing values", "decision": fill_choice.split(" · ", 1)[1],
     "evidence": line(fill_choice, reference) if fill_choice != reference else "reference kept"},
    {"block": "history flags", "decision": "added" if flags_added else "not added",
     "evidence": line(flags_name, fill_choice)},
    {"block": "outliers", "decision": f"no row deleted; log1p on {FINAL['log1p'] or 'nothing'}",
     "evidence": line(outlier_choice, "after §2 · missing values")
     if outlier_choice != "after §2 · missing values" else "no treatment kept"},
    {"block": "encoding + scaling", "decision": f"{FINAL['encoding']} + {FINAL['scaler']}",
     "evidence": line(encoding_choice, "after §3 · outliers") if encoding_choice != "after §3 · outliers"
     else "current kept"},
    {"block": "final recipe vs V01", "decision": "all blocks together", "evidence": line(current, V01)},
]).set_index("block")

**Conclusions**

| Block | Decision | Evidence (paired ROC AUC, 3 folds) |
|---|---|---|
| Missing values | **iterative / own level** | +0.0010 ± 0.0004 vs median / own level, 3/3; all plans within ±0.0015 |
| History flags | **not added** | −0.0002 ± 0.0008 |
| Outliers | **no row deleted; log1p on the 3 day intervals** | +0.0008 ± 0.0003, 3/3; winsorize −0.0031, log1p on counts −0.0048 |
| Encoding + scaling | **count + standard** | +0.0059 ± 0.0048, 3/3 (inside the noise); chosen because one-hot encodes 209 unseen test rows as the reference level |
| **Final recipe vs V01** | | **+0.0106 ± 0.0029, 3/3; honest F1 0.787 vs 0.771** |

1. **Missing values and outliers matter little on this data:** each gains about a thousandth of ROC AUC. The structural gaps are already explained by `prior_purchase_count`, and the extreme values are real, loyal customers.
2. **The encoding is the decision that matters most:** +0.0059 ROC AUC, against about +0.001 for each of the other blocks, and the honest F1 of the final recipe is +0.016 above V01. The validation could not prove it beyond 2 SEM, so it was decided by what happens to levels never seen in training, the situation of 21% of the test rows.
3. **The recipe was chosen through a logistic regression:** it helps gradient boosting too, but KNN gets worse.

**For `05_features`:** how `transport_method` enters the model (80 levels, drives the fold-3 pattern); zero-inflated columns such as `discount_global` and `total_shipping` (0/1 flags); the history flags with tree models.

**For `06_models`:** check the scaler and encoder for distance-based models (KNN).

The recipe is in `src/preprocessing.py` (`final_recipe()`), and the rows for `docs/decisions.md` are printed in §6.